# Financial Analysis Demo with Cash

This notebook demonstrates the capabilities of the `cash` library for caching expensive data science operations. We will work with a large, deterministically generated financial dataset.

In [1]:
import cash
import pandas as pd
import numpy as np
import time
import os
print(os.getcwd())
#os.chdir(r'C:\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples')

c:\Users\Philipp\Downloads\oldpc\nvme\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples


In [2]:
%cash_on
#%cash_debug on

✅ Cash enabled. Your computations will be cached automatically.
   Run %cash_help for available commands.
   Found existing cache with 338 entries.
[Tip] Cash reads upstream cells from the saved notebook file.
   Save (Ctrl+S) after editing upstream cells, or enable auto-save:
   Settings -> "files.autoSave": "afterDelay"


## 1. Data Loading
Loading a large CSV file can be slow. With `cash`, this operation is cached after the first run.

In [7]:
print(os.getcwd())
# Ensure the data exists (it should have been generated by generate_financial_data.py)
data_path = 'large_financial_data.csv'
#if not os.path.exists(data_path):
    #print("Data file not found! Please run generate_financial_data.py first.")
#else:
print("Loading data...")
# This read_csv call will be cached
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
print(df.head())

c:\Users\Philipp\Downloads\oldpc\nvme\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples
Loading data...
                 Date Ticker        Open        High         Low       Close  \
0 2020-01-01 00:00:00   AAPL  100.496814  100.835158   98.367369   99.389654   
1 2020-01-01 00:01:00  GOOGL  100.358650  100.601660   99.365223   98.589288   
2 2020-01-01 00:02:00   MSFT  101.006438  103.319689  100.747918  100.668630   
3 2020-01-01 00:03:00   AMZN  102.529568  102.929740   99.570182  103.031946   
4 2020-01-01 00:04:00   TSLA  102.295515  104.024736  101.675820  102.739018   

   Volume  
0  611180  
1  410314  
2   12462  
3  133239  
4  968875  


## 2. Preprocessing
Basic sorting and cleaning.

In [ ]:
print("Sorting data...")
t0 = time.time()
df = df.sort_values(by=['Ticker', 'Date'])
print(f"Sorted in {time.time() - t0:.2f}s")

In [5]:
df #print

,Date,Ticker,Open,High,Low,Close,Volume
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122
...,...,...,...,...,...,...,...
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000018,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000019,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000020,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156


## 3. Heavy Computation (Statement-wise Caching)
Here we perform multiple heavy calculations using custom rolling window functions. These are significantly slower than vectorized pandas operations, making them perfect candidates for caching. `cash` caches these statement-wise.

**Try this:** Run the cell once. Then change the window size in the first statement (SMA) and run it again. You'll see the second statement (Voladj) loads instantly from cache!

In [3]:
print("Calculating Volatility Adjusted Mean (Statement 1)....")
t0 = time.time()
# Heavy operation 1: Another slow rolling application
df['VolAdj_20'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=20).apply(lambda y: np.mean(y) / (np.std(y) + 1e-6), raw=True))
print(f"VolAdj calculated in {time.time() - t0:.2f}s")

print("Calculating Weighted SMA (Statement 2)...")
t0 = time.time()
# Heavy operation 2: Custom weighted mean using apply() (slow)
def custom_weighted_mean(x):
    weights = np.arange(1, len(x) + 1)
    return np.sum(x * weights) / np.sum(weights)

df['SMA_71'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=50).apply(custom_weighted_mean, raw=True))
print(f"SMA calculated  in {time.time() - t0:.2f}s")


df

Calculating Volatility Adjusted Mean (Statement 1)....
VolAdj calculated in 0.02s
Calculating Weighted SMA (Statement 2)...
SMA calculated  in 0.08s


,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_71
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180,NaN,NaN
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078,NaN,NaN
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047,NaN,NaN
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594,NaN,NaN
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.276968,0.435355
1000018,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.411779
1000019,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.388156
1000020,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.364842


In [3]:
#print(x)
df

,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_71
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180,NaN,NaN
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078,NaN,NaN
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047,NaN,NaN
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594,NaN,NaN
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.276968,0.435355
1000018,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.411779
1000019,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.388156
1000020,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.364842


In [ ]:
%cash_provenance df --graph   

NameError: name 'AmbiguousCellError' is not defined

## 4. More Metrics
Adding RSI calculation in a separate cell.

In [5]:
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

print("Calculating RSI...")
t0 = time.time()
df['RSI'] = df.groupby('Ticker')['Close'].transform(calculate_rsi)
print(f"RSI calculated in {time.time() - t0:.2f}s")
df

Calculating RSI...
RSI calculated in 0.04s


,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_71,RSI
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180,NaN,NaN,NaN
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078,NaN,NaN,NaN
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047,NaN,NaN,NaN
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594,NaN,NaN,NaN
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.276968,0.435355,NaN
1000018,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.411779,NaN
1000019,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.388156,NaN
1000020,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,26647.418221,0.364842,NaN


## 5. Aggregation & Analysis

In [ ]:
summary = df.groupby('Ticker').agg({
    'Close': ['mean', 'std'],
    'Volume': 'sum',
    'RSI': 'mean',
    'SMA_50': 'last'
})
print(summary)

In [5]:
print("hi")
print(df)

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X21sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 01:12:30.732464
[ENSURE_STATE_DEBUG] Cell code: print("hi")
print(df)...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'df', 'print'}
[ENSURE_STATE_DEBUG] Analyzed outputs: set()
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 17.28ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 17.29ms
[TIMING_PROXY] Start executing statements...
hi
                       Date Ticker        Open        High         Low  \
0       2020-01-01 00:00:00   AAPL  100.496814  100.835158   98.367369   
5       2020-01-01 00:05:00   AAPL  102.061478  104.873044   98.079078   
10      2020-01-01 00:10:00   AAPL  104.018293  106.607716  103.477609   
15      2020-01-01 00:1

## 6. Loop Caching Demo

Cash can now cache **individual iterations** of loops! Each iteration is cached separately based on:
- The loop variable value
- Dependencies accessed in that iteration

This means if you change earlier iterations, later iterations that are independent can still be restored from cache.

In [5]:
# Processing each ticker separately in a loop
# Each iteration is cached independently!
ticker_stats = {}
print(df)
a = []

for ticker in ["TSLA", "GOOGL", "AAPL", "AMZN"]:
    ticker_data = df[df["Ticker"] == ticker]
    stats = {
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
    }
    ticker_stats[ticker] = stats
    print(f"{ticker}: mean={stats['mean_close'][0]:.2f}, std={stats['std_close']:.2f}")
    a.append(ticker)

print(ticker_stats.keys())
print("\nDone processing all tickers!")


[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X23sZmlsZQ%3D%3D


TYPE,CONTENT,STORAGE,TIME
⚡ Restored,ticker_statsticker_stats = {},← RAM,Saved 0.00s
⚡ Restored,No outputsprint(df),← RAM,Saved 0.00s
⚡ Restored,aa = [],← RAM,Saved 0.00s
⚙️ Loop,"▶ 🔁 ticker ∈ [TSLA, GOOGL, AAPL, AMZN] (All 4 computed, 5 stmts)",→ RAM,0.24s (Saved 6.02s)
⚡,▸ ticker_data = df[df['Ticker'] == ticker] (all cached),← RAM,0.06s (↑0.0s)
⚡,cached — ticker=TSLA,← RAM,0.014s (↑0.0s)
⚡,cached — ticker=GOOGL,← RAM,0.014s (↑0.0s)
⚡,cached — ticker=AAPL,← RAM,0.014s (↑0.0s)
⚡,cached — ticker=AMZN,← RAM,0.014s (↑0.0s)
⚡,"▸ stats = {'mean_close': [ticker_data['Close'].mean() for i in range(10000)], 'std_close': ticker_data['Close'].std(), 'mi… (all cached)",← RAM,0.05s (↑6.0s)


[TIMING_PROXY] Start cached_run_cell: 10:54:21.986005
[ENSURE_STATE_DEBUG] Cell code: # Processing each ticker separately in a loop
# Ea...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'range', 'df', 'print'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'a', 'stats', 'ticker_data', 'ticker_stats', 'ticker'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 4.51ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 4.52ms
[TIMING_PROXY] Start executing statements...
                       Date Ticker        Open        High         Low  \
0       2020-01-01 00:00:00   AAPL  100.496814  100.835158   98.367369   
5       2020-01-01

In [6]:
ticker_data

Cash: Auto-executing upstream statement: ticker_stats = {}
Cash: Auto-executing upstream statement: a = []
Cash: Auto-executing upstream statement: for ticker in ['TSLA', 'GOOGL', 'AAPL', ...
TSLA: mean=78.25, std=120.39
GOOGL: mean=78.27, std=120.39
AAPL: mean=78.26, std=120.40
AMZN: mean=78.27, std=120.40


,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_71,RSI
3,2020-01-01 00:03:00,AMZN,102.529568,102.929740,99.570182,103.031946,133239,NaN,NaN,NaN
8,2020-01-01 00:08:00,AMZN,103.938951,105.828290,100.001502,105.048034,254901,NaN,NaN,NaN
13,2020-01-01 00:13:00,AMZN,101.881546,103.248859,100.452151,101.716643,132808,NaN,NaN,NaN
18,2020-01-01 00:18:00,AMZN,97.988232,98.746731,94.491528,99.501512,944457,NaN,NaN,NaN
23,2020-01-01 00:23:00,AMZN,96.459081,97.231260,94.307023,95.938781,406026,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
999978,2021-11-25 10:18:00,AMZN,1.000000,3.108617,-1.428544,0.233533,434109,1.202176,0.981772,42.043465
999983,2021-11-25 10:23:00,AMZN,1.000000,2.574597,-0.773791,0.203965,939126,1.148672,0.946020,52.456797
999988,2021-11-25 10:28:00,AMZN,1.000000,2.839892,0.461452,1.826587,706917,1.180048,0.976062,57.595581
999993,2021-11-25 10:33:00,AMZN,1.000000,1.512479,0.475784,0.771204,834392,1.147620,0.965136,44.856435


In [5]:
{
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
}

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#Y114sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 14:15:37.209718
[ENSURE_STATE_DEBUG] Cell code: {
        "mean_close": [ticker_data["Close"].mean...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'ticker_data', 'range'}
[ENSURE_STATE_DEBUG] Analyzed outputs: set()
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 9.49ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 9.53ms
[TIMING_PROXY] Start executing statements...
[TIMING_PROXY] PROXY TOTAL: 424.3ms
[TIMING_PROXY] Badge init: 6.4ms
[TIMING_PROXY] Upstream check: 9.5ms
[TIMING_PROXY] Badge progress renders: 10.3ms


{'mean_close': [nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,

[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#Y114sZmlsZQ%3D%3D


In [ ]:
ticker_stats.keys()
a

### Conditional Caching

Cash also caches **branches of if statements** - only the executed branch is stored, making cache keys more precise.

In [7]:
# Conditional processing based on data size
row_count = len(df)

if row_count > 500000:
    # Heavy processing for large datasets
    sample = df.sample(n=10000, random_state=42)
    report_type = "sampled"
    print(f"Large dataset ({row_count:,} rows) - using sampled analysis")
else:
    # Full processing for smaller datasets
    sample = df.copy()
    report_type = "full"
    print(f"Small dataset ({row_count:,} rows) - using full analysis")

print(f"Report type: {report_type}, sample size: {len(sample):,}")

Large dataset (1,000,022 rows) - using sampled analysis
Report type: sampled, sample size: 10,000


### Nested Loops Example

Even nested loops work - each combination of outer/inner loop variables gets its own cache entry.

In [8]:
# Compute metrics for multiple tickers across different time windows
windows = [10, 20, 50]
tickers = ["AAPL", "GOOGL"]

results = {}
for ticker in tickers:
    results[ticker] = {}
    ticker_data = df[df["Ticker"] == ticker]["Close"]
    for window in windows:
        sma = ticker_data.rolling(window=window).mean().iloc[-1]
        results[ticker][f"SMA_{window}"] = sma
        print(f"{ticker} SMA-{window}: {sma:.2f}")

print("\nAll window calculations complete!")

AAPL SMA-10: 0.83
AAPL SMA-20: 1.02
AAPL SMA-50: 0.84
GOOGL SMA-10: 0.86
GOOGL SMA-20: 1.22
GOOGL SMA-50: 1.12

All window calculations complete!


In [ ]:
for a, b in zip([1, 2, 3, 4, 5, 6, 7], ['x', 'y', 'z', 'u', 'v', 'w', 't']):
    print(f"{a} - {b}")

In [ ]:
try:
    print("hi")
    asdf = 235
    raise ValueError("This is a test error")
    x = 123
except ValueError as e:
    print(f"Value error occurred: {e}")
    time.sleep(1)  # Simulate some cleanup time

In [ ]:
c = 122

In [ ]:

def f(a):
    return c + a

x = {'b': 123}
x['a'] = f(0)
print(x)

In [ ]:
x

In [ ]:
import sys
sys.path.append("examples")
import metrics
print("Testing metrics module...")
print(metrics.increment(5))

In [ ]:
class Test:
    def __init__(self):
        self.x = 1
        self.y = 2

a = Test()
print(a.x)

a.x = 126

print(a.x)

In [ ]:
a.x = 1290

In [ ]:
print(a.x)
a.x

In [ ]:
print("hi")
raise ValueError("This is a test error to demonstrate error handling in the notebook.")
print("ho")

In [7]:
@cash.cache
def dep(a):
    import time
    time.sleep(1)
    return a + 1

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X51sZmlsZQ%3D%3D


TYPE,CONTENT,STORAGE,TIME


Cash auto-caching failed: name 'UpstreamChecker' is not defined. Falling back to normal execution.


[TIMING_PROXY] Start cached_run_cell: 23:27:21.336325
[ENSURE_STATE_DEBUG] Cell code: @cash.cache
def dep(a):
    import time
    time.s...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'cash'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'dep'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X51sZmlsZQ%3D%3D


In [5]:
@cash.cache
def fun(a, b):
    import time
    time.sleep(1)
    return a + b + dep(a)

x = 6
[fun(x, i % 3) for i in range(52)]

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X52sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:41:08.079143
[ENSURE_STATE_DEBUG] Cell code: @cash.cache
def fun(a, b):
    import time
    tim...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'dep', 'cash', 'range'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'x', 'fun'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 5.05ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 5.07ms
[TIMING_PROXY] Start executing statements...
[TIMING_PROXY] PROXY TOTAL: 20.7ms
[TIMING_PROXY] Badge init: 3.2ms
[TIMING_PROXY] Upstream check: 5.0ms
[TIMING_PROXY] Badge progress renders: 13.1ms


[13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13,
 14,
 15,
 13]

[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X52sZmlsZQ%3D%3D


In [4]:
import metrics

print(metrics.super_fun(10))
sum([metrics.fun(x, i % 3) for i in range(51)])

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X53sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 14:32:22.678749
[ENSURE_STATE_DEBUG] Cell code: import metrics

print(metrics.super_fun(10))
sum([...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'x', 'range', 'print', 'sum'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'metrics'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[STATE] 'sum' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 5.16ms
[TIMING_PROXY] Total restore time: 0.02ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 5.17ms
[TIMING_PROXY] Start executing statements...
22
[TIMING_PROXY] PROXY TOTAL: 6061.8ms
[TIMING_PROXY] Badge init: 11.0ms
[TIMING_PROXY] Upstream check: 5.1ms
[TIMING_PROXY] Badge prog

918

[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X53sZmlsZQ%3D%3D


In [ ]:
@cash.cache
def super_fun(df):
    import time
    time.sleep(1)  # Simulate a heavy operation
    return df.iloc[1]

In [11]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6]
})
print(super_fun(df), "", "")

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X55sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 14:33:14.260996
[ENSURE_STATE_DEBUG] Cell code: df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'pd', 'super_fun', 'print'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'df'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 4.50ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 4.52ms
[TIMING_PROXY] Start executing statements...
A    2
B    5
Name: 1, dtype: int64  
[TIMING_PROXY] PROXY TOTAL: 15.0ms
[TIMING_PROXY] Badge init: 2.9ms
[TIMING_PROXY] Upstream check: 4.5ms
[TIMING_PROXY] Badge progress renders: 8.4ms
[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Use